RunnablePassthrough는 데이터를 전달하는 역할을 합니다. 이 클래스는 invoke() 메서드를 통해 입력된 데이터를 그대로 반환 합니다.

이는 데이터를 변경하지 않고 파이프라인의 다음 단계로 전달하는 데 사용될 수 있습니다.

RunnablePassthrough 는 다음과 같은 시나리오에서 유용할 수 있습니다.

데이터를 변환하거나 수정할 필요가 없는 경우
파이프라인의 특정 단계를 건너뛰어야 하는 경우
디버깅 또는 테스트 목적으로 데이터 흐름을 모니터링해야 하는 경우
이 클래스는 Runnable 인터페이스를 구현하므로, 다른 Runnable 객체와 함께 파이프라인에서 사용될 수 있습니다

In [17]:
from dotenv import load_dotenv

load_dotenv()

True

In [18]:
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("CH13-LCEL-Advanced")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH13-LCEL-Advanced


데이터 전달하기
RunnablePassthrough 는 입력을 변경하지 않고 그대로 전달하거나 추가 키를 더하여 전달할 수 있습니다.

일반적으로 RunnableParallel 과 함께 사용되어 데이터를 맵의 새로운 키에 할당하는 데 활용됩니다.

RunnablePassthrough() 를 단독으로 호출하면 단순히 입력을 받아 그대로 전달합니다.

assign과 함께 호출된 RunnablePassthrough(RunnablePassthrough.assign(...))는 입력을 받아 assign 함수에 전달된 추가 인자를 더합니다.

-RunnableParallel 클래스를 사용하여 병렬로 실행 가능한 작업을 정의 합니다.
-passed 속성에는 RunnablePassthrough 인스턴스를 할당하여 입력을 그대로 반환하도록 설정합니다.
-extra 속성에는 RunnablePassthrough.assign() 메서드를 사용하여 입력의 "num" 값에 3을 곱한 결과를 "mult" 키에 할당하는 작업을 정의합니다.
-modified 속성에는 람다 함수를 사용하여 입력의 "num" 값에 1을 더하는 작업을 정의합니다.
-runnable.invoke() 메서드를 호출하여 {"num": 1} 입력으로 병렬 작업을 실행합니다.

In [19]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

runnable = RunnableParallel(
    passed=RunnablePassthrough(),
    extra=RunnablePassthrough.assign(mult=lambda x: x["num"]*3),
    modified=lambda x: x["num"] + 1,
)

runnable.invoke({"num": 1})

{'passed': {'num': 1}, 'extra': {'num': 1, 'mult': 3}, 'modified': 2}

In [20]:
r = RunnablePassthrough.assign(mult=lambda x:x["num"]*3)
r.invoke({"num": 1})

{'num': 1, 'mult': 3}

In [21]:
#검색기 예제

from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

vectorstore = FAISS.from_texts(
    [
        "테디는 랭체인 주식회사에서 근무를 하였습니다.",
        "셜리는 테디와 같은 회사에서 근무하였습니다.",
        "테디의 직업은 개발자입니다.",
        "셜리의 직업은 디자이너입니다.",  
    ],
    embedding=OpenAIEmbeddings(),
)

#벡터 저장소를 검색기로 사용
retriever = vectorstore.as_retriever()
#템플렛 정의
template = """Answer te question based only on the following context:
{context}

Question: {question}"""

#템플릿으로부터 채팅 프롬프트를 생성
prompt = ChatPromptTemplate.from_template(template)

model = ChatOpenAI(model_name="gpt-4o-mini")

def format_docs(docs):
    return "\n".join([doc.page_content for doc in docs])

retrieval_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [22]:
retrieval_chain.invoke("테디의 직업은 무엇입니까?")

'테디의 직업은 개발자입니다.'

In [23]:
#runnables 구조 검토

# !pip install -qU faiss-cpu tiktoken
# !pip install -qU grandalf

In [24]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

vectorstore = FAISS.from_texts(
    ["Teddy is an AI engineer who loves programming!"],
    embedding=OpenAIEmbeddings(),
)

retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context:
{context}

Question: {question}"""

prompt = ChatPromptTemplate.from_template(
    template
)

model = ChatOpenAI(model="gpt-4o-mini")

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [25]:
chain.get_graph().nodes

{'ce256de8a86042028aee6367cae72f73': Node(id='ce256de8a86042028aee6367cae72f73', name='Parallel<context,question>Input', data=<class 'langchain_core.runnables.base.RunnableParallel<context,question>Input'>, metadata=None),
 '2f2b04c42abf4213a0236fc4dc28e30a': Node(id='2f2b04c42abf4213a0236fc4dc28e30a', name='Parallel<context,question>Output', data=<class 'langchain_core.utils.pydantic.RunnableParallel<context,question>Output'>, metadata=None),
 '249115b82c5c42f2b7e820ab63823a77': Node(id='249115b82c5c42f2b7e820ab63823a77', name='VectorStoreRetriever', data=VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000209A661F3E0>, search_kwargs={}), metadata=None),
 '0191253f111e4709bd1a61088ee37b86': Node(id='0191253f111e4709bd1a61088ee37b86', name='Passthrough', data=RunnablePassthrough(), metadata=None),
 'db6978856f684382a022fc11dd601019': Node(id='db6978856f684382a022fc11dd601019', name='ChatPromptTemplate', dat

In [26]:
# 체인의 그래프를 ASCII 형식으로 출력합니다.
chain.get_graph().print_ascii()

           +---------------------------------+         
           | Parallel<context,question>Input |         
           +---------------------------------+         
                    **               **                
                 ***                   ***             
               **                         **           
+----------------------+              +-------------+  
| VectorStoreRetriever |              | Passthrough |  
+----------------------+              +-------------+  
                    **               **                
                      ***         ***                  
                         **     **                     
           +----------------------------------+        
           | Parallel<context,question>Output |        
           +----------------------------------+        
                             *                         
                             *                         
                             *                  

In [27]:
chain.get_prompts()

[ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n{context}\n\nQuestion: {question}'), additional_kwargs={})])]

In [28]:
#runnable lambda : 사용자 정의 함수를 실행 

from operator import itemgetter

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

def length_function(text):
    return len(text)

def _multiple_length_function(text1, text2):
    return len(text1) * len(text2)

def multiple_length_function(
        _dict,
):
    return _multiple_length_function(_dict["text1"], _dict["text2"])


# 프롬프트 템플릿 생성
prompt = ChatPromptTemplate.from_template("what is {a} + {b}?")
# ChatOpenAI 모델 초기화
model = ChatOpenAI()

# 프롬프트와 모델을 연결하여 체인 생성
chain1 = prompt | model

# 체인 구성
chain = (
    {
        "a": itemgetter("input_1") | RunnableLambda(length_function),
        "b": {"text1": itemgetter("input_1"), "text2": itemgetter("input_2")}
        | RunnableLambda(multiple_length_function),
    }
    | prompt
    | model
    | StrOutputParser()
)

In [29]:
chain.invoke({"input_1": "bar", "input_2": "gah"})

'3 + 9 equals 12.'

In [30]:
chain.get_graph().print_ascii()

                      +--------------------+                        
                      | Parallel<a,b>Input |                        
                      +--------------------+                        
                       ***                 ****                     
                    ***                        ****                 
                  **                               ****             
+----------------------------+                         **           
| Parallel<text1,text2>Input |                          *           
+----------------------------+                          *           
          **        **                                  *           
        **            **                                *           
       *                *                               *           
+--------+          +--------+                          *           
| Lambda |          | Lambda |                          *           
+--------+          +--------+    

In [31]:
#runnableconfig

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableConfig
import json

def parse_or_fix(text: str, config: RunnableConfig):
    fixing_chain = (
        ChatPromptTemplate.from_template(
            "Fix the following text: \n\ntext\n{input}\n\nError: {error}\n\n"
            "Don't narrate, just respond wth the fixed data."
        )
        | ChatOpenAI()
        | StrOutputParser()
    )

    for _ in range(3):
        try:
            return json.loads(text)
        except Exception as e:
            text = fixing_chain.invoke({"input": text, "error": e}, config)
            print(f"config: {config}")
        return "Failed to parse"

In [32]:
from langchain_community.callbacks.manager import get_openai_callback

with get_openai_callback() as cb:
    output = RunnableLambda(parse_or_fix).invoke(
        input="{foo:: bar}",
        config={"tags": ["my-tag"], "callbacks": [cb]},
    )

    print(f"\n\n수정한결과:\n{output}")

config: {'tags': ['my-tag'], 'metadata': {}, 'callbacks': <langchain_core.callbacks.manager.CallbackManager object at 0x00000209A7E180B0>, 'recursion_limit': 25, 'configurable': {}}


수정한결과:
Failed to parse


In [ ]:
#runnablebranch: 입력값에 따라 여러 갈래 중 하나를 선택해서 실행하는 Runnable

from langchain_openai import ChatOpenAI
